In [ ]:
USE DATABASE AI_DEMOS;
DROP SCHEMA IF EXISTS WAREHOUSE_OPTIMIZATION;
DROP SCHEMA IF EXISTS WAREHOUSE_OPTIMIZATION_FEATURE_STORE;
DROP SCHEMA IF EXISTS WAREHOUSE_OPTIMIZATION_MODEL_REGISTRY;

# Imports

Set up the environment by installing required packages and importing libraries for time-series forecasting, feature engineering, and Snowflake ML operations.

In [ ]:
%pip install --q -e ../ ipywidgets mlforecast lightgbm utilsforecast

In [ ]:
import copy
import pandas as pd
import numpy as np
from datetime import date, timedelta

import lightgbm as lgb
from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean, RollingStd, ExpandingMean
from utilsforecast.losses import rmse, smape
from utilsforecast.evaluation import evaluate

from snowflake.snowpark import functions as F
from snowflake.snowpark.context import get_active_session

from snowflake.ml.experiment import ExperimentTracking
from snowflake.ml.feature_store import FeatureStore, CreationMode, Entity, FeatureView
from snowflake.ml.registry import Registry
from snowflake.ml.model.custom_model import CustomModel, ModelContext
from snowflake.ml.model import custom_model, model_signature, type_hints
from snowflake.ml.monitoring.entities.model_monitor_config import ModelMonitorConfig, ModelMonitorSourceConfig

import demo_functions

session = get_active_session()

database = 'AI_DEMOS'
schema = 'WAREHOUSE_OPTIMIZATION'
warehouse = 'AI_WH'

session.use_database(database)

demo_functions.setup(session, schema)

# 1 - Setup Feature Store and Model Registry

Initialize the Snowflake Feature Store for managing and serving features, and the Model Registry for versioning, deploying, and monitoring ML models.

In [ ]:
my_feature_store = FeatureStore(
    session=session,
    database=database,
    name=f"{schema}_FEATURE_STORE",
    default_warehouse=warehouse,
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST,
)

my_model_registry = Registry(
    session=session,
    database_name=database,
    schema_name=f'{schema}_MODEL_REGISTRY',
    options={'enable_monitoring': True}
)

# 2 - Explore Data

Examine the warehouse data including static attributes, time-varying properties, and daily operational metrics. Visualize utilization patterns, seasonal trends, and distribution across warehouses.

In [ ]:
warehouse_dim = session.table(f'{database}.{schema}.WAREHOUSES')
print('Warehouses (static):')
display(warehouse_dim.limit(5))

warehouse_attrs = session.table(f'{database}.{schema}.WAREHOUSE_ATTRIBUTES')
print('Warehouse Attributes (time-varying):')
display(warehouse_attrs.limit(5))

warehouse_ops = session.table(f'{database}.{schema}.WAREHOUSE_OPERATIONS')
print('Warehouse Operations:')
display(warehouse_ops.limit(5))

In [ ]:
viz_df = (
    warehouse_ops
        .join(warehouse_dim, on='WAREHOUSE_ID', how='left')
        .order_by('WAREHOUSE_ID', 'DATE')
        .to_pandas()
)

demo_functions.plot_utilization_timeseries(viz_df, warehouse_ids=['WH_001','WH_002','WH_003','WH_004','WH_005'])

In [ ]:
demo_functions.plot_seasonal_patterns(viz_df)

In [ ]:
attrs_df = warehouse_attrs.to_pandas()
demo_functions.plot_warehouse_attributes_timeline(attrs_df, warehouse_ids=['WH_001','WH_002','WH_003'])

In [ ]:
demo_functions.plot_utilization_distribution(viz_df)

# 3 - Register Warehouse Features in Feature Store

The Feature Store manages non-time-series features via three Feature Views:

1. **Static Feature View** (no `timestamp_col`): Immutable warehouse attributes — region and type. One row per warehouse.
2. **Dynamic Feature View** (with `timestamp_col='EFFECTIVE_DATE'`): Time-varying physical attributes — capacity, loading docks, staffing level. The Feature Store automatically performs point-in-time correct lookups.
3. **Operational Feature View** (with `timestamp_col='DATE'`): Daily operational metrics — inbound/outbound shipments, temperature, active clients. These are causal drivers of utilization.

In [ ]:
warehouse_entity = Entity(
    name="WAREHOUSE",
    join_keys=["WAREHOUSE_ID"],
    desc="Unique Warehouse ID"
)

my_feature_store.register_entity(warehouse_entity)

In [ ]:
# Static Features
static_features_df = (
    warehouse_dim
    .select('WAREHOUSE_ID', 'REGION', 'WAREHOUSE_TYPE')
    .with_column('IS_EMEA', F.iff(F.col('REGION') == 'EMEA', 1, 0))
    .with_column('IS_APAC', F.iff(F.col('REGION') == 'APAC', 1, 0))
    .with_column('IS_AMERICAS', F.iff(F.col('REGION') == 'Americas', 1, 0))
    .with_column('IS_COLD_STORAGE', F.iff(F.col('WAREHOUSE_TYPE') == 'Cold Storage', 1, 0))
    .with_column('IS_HAZMAT', F.iff(F.col('WAREHOUSE_TYPE') == 'Hazmat', 1, 0))
    .with_column('IS_AMBIENT', F.iff(F.col('WAREHOUSE_TYPE') == 'Ambient', 1, 0))
    .drop('REGION', 'WAREHOUSE_TYPE')
)

warehouse_static_fv = FeatureView(
    name='WAREHOUSE_STATIC_FEATURES',
    entities=[warehouse_entity],
    feature_df=static_features_df,
    refresh_freq='1 day',
    desc='Static warehouse attributes: region, type (one-hot encoded). Never change over time.'
)

warehouse_static_fv = my_feature_store.register_feature_view(
    feature_view=warehouse_static_fv,
    version='1',
    overwrite=True
)

print('Static features (one row per warehouse):')
display(static_features_df.limit(5))

In [ ]:
# Dynamic Features
dynamic_features_df = warehouse_attrs.select(
    'WAREHOUSE_ID',
    'EFFECTIVE_DATE',
    'TOTAL_CAPACITY_SQM',
    'NUM_LOADING_DOCKS',
    'STAFFING_LEVEL',
)

warehouse_dynamic_fv = FeatureView(
    name='WAREHOUSE_DYNAMIC_FEATURES',
    entities=[warehouse_entity],
    feature_df=dynamic_features_df,
    timestamp_col='EFFECTIVE_DATE',
    refresh_freq='1 day',
    desc='Time-varying warehouse attributes: capacity, loading docks, staffing level. Uses timestamp_col for point-in-time correct lookups.'
)

warehouse_dynamic_fv = my_feature_store.register_feature_view(
    feature_view=warehouse_dynamic_fv,
    version='1',
    overwrite=True
)

print('Dynamic features (multiple rows per warehouse, SCD-style):')
display(dynamic_features_df.order_by('WAREHOUSE_ID', 'EFFECTIVE_DATE').limit(10))

In [ ]:
# Operational Features
operational_features_df = warehouse_ops.select(
    'WAREHOUSE_ID',
    'DATE',
    'INBOUND_SHIPMENTS',
    'OUTBOUND_SHIPMENTS',
    'TEMPERATURE_C',
    'NUM_ACTIVE_CLIENTS',
)

warehouse_operational_fv = FeatureView(
    name='WAREHOUSE_OPERATIONAL_FEATURES',
    entities=[warehouse_entity],
    feature_df=operational_features_df,
    timestamp_col='DATE',
    refresh_freq='1 day',
    desc='Daily operational metrics: inbound/outbound shipments, temperature, active clients. Causal drivers of utilization.'
)

warehouse_operational_fv = my_feature_store.register_feature_view(
    feature_view=warehouse_operational_fv,
    version='1',
    overwrite=True
)

print(f'Registered: {warehouse_operational_fv.name} (operational, timestamp_col=DATE)')
display(operational_features_df.order_by('WAREHOUSE_ID', 'DATE').limit(5))

# 4 - Generate Training Dataset

Use `generate_dataset()` to create an immutable snapshot joining all Feature Views to the operations spine. The dynamic FV uses point-in-time correct lookups via `spine_timestamp_col='DATE'` to prevent data leakage.

In [ ]:
training_cutoff = (date.today() - timedelta(days=demo_functions.FORECAST_HORIZON_DAYS)).isoformat()

spine_df = (
    session.table(f'{database}.{schema}.WAREHOUSE_OPERATIONS')
    .filter(F.col('DATE') <= training_cutoff)
    .select('WAREHOUSE_ID', 'DATE', 'UTILIZATION_PCT')
    .distinct()
)

print(f'Training cutoff: {training_cutoff}')
print('Spine DataFrame:')
display(spine_df.limit(5))

In [ ]:
training_dataset = my_feature_store.generate_dataset(
    name=f'{database}.{schema}_MODEL_REGISTRY.WAREHOUSE_CAPACITY_DATASET_V1',
    spine_df=spine_df,
    features=[warehouse_static_fv, warehouse_dynamic_fv, warehouse_operational_fv],
    spine_timestamp_col='DATE',
    spine_label_cols=['UTILIZATION_PCT'],
    desc='Immutable training dataset for warehouse capacity forecasting V1.'
)

training_dataset_sf = training_dataset.read.to_snowpark_dataframe()
print(f'Dataset created: {training_dataset_sf.count()} rows')
display(training_dataset_sf.limit(5).to_pandas())

# 5 - Prepare Data for mlforecast

Transform the training dataset into the format required by mlforecast: rename columns to standard names (`unique_id`, `ds`, `y`), split into train/validation sets, and identify feature columns by type.

In [ ]:
training_dataset_sf.order_by('WAREHOUSE_ID', 'DATE').to_pandas().columns

In [ ]:
ops_df = training_dataset_sf.order_by('WAREHOUSE_ID', 'DATE').to_pandas()

static_feature_cols = ['IS_EMEA', 'IS_APAC', 'IS_AMERICAS', 'IS_COLD_STORAGE', 'IS_HAZMAT', 'IS_AMBIENT']
dynamic_feature_cols = ['TOTAL_CAPACITY_SQM', 'NUM_LOADING_DOCKS', 'STAFFING_LEVEL']
operational_feature_cols = ['INBOUND_SHIPMENTS', 'OUTBOUND_SHIPMENTS', 'TEMPERATURE_C', 'NUM_ACTIVE_CLIENTS']
all_feature_cols = static_feature_cols + dynamic_feature_cols + operational_feature_cols

ops_df = ops_df.rename(columns={
    'WAREHOUSE_ID': 'unique_id',
    'DATE': 'ds',
    'UTILIZATION_PCT': 'y'
})

ops_df['ds'] = pd.to_datetime(ops_df['ds'])

#if 'EFFECTIVE_DATE' in ops_df.columns:
#    ops_df = ops_df.drop(columns=['EFFECTIVE_DATE'])
#if 'DATE_1' in ops_df.columns:
#    ops_df = ops_df.drop(columns=['DATE_1'])

valid = ops_df.groupby('unique_id', observed=True).tail(demo_functions.FORECAST_HORIZON_DAYS)
train = ops_df.drop(valid.index)

print(f'Total data shape: {ops_df.shape}')
print(f'Train shape: {train.shape}, Validation shape: {valid.shape}')
print(f'Date range: {ops_df["ds"].min()} to {ops_df["ds"].max()}')
print(f'Number of series: {ops_df["unique_id"].nunique()}')
print(f'Static feature columns: {static_feature_cols}')
print(f'Dynamic feature columns: {dynamic_feature_cols}')
print(f'Operational feature columns: {operational_feature_cols}')
display(ops_df.head())

# 6 - Define Training Function

Configure the MLForecast pipeline with LightGBM, including lag features, rolling statistics, and date features. Define a reusable training function that performs cross-validation and logs metrics to experiment tracking.

In [ ]:
exp = ExperimentTracking(session=session)
exp.set_experiment('WAREHOUSE_CAPACITY_PLANNING')

h = demo_functions.FORECAST_HORIZON_DAYS

params = {
    'init': {
        'models': {
            'LGBMRegressor': lgb.LGBMRegressor(
                n_estimators=500,
                max_depth=6,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                min_child_weight=5,
                random_state=42,
                verbosity=-1,
                n_jobs=-1,
            ),
        },
        'freq': 'D',
        'lags': [1, 7, 14, 28],
        'lag_transforms': {
            1: [ExpandingMean()],
            7: [RollingMean(window_size=7), RollingStd(window_size=7)],
            14: [RollingMean(window_size=14), RollingStd(window_size=14)],
            28: [RollingMean(window_size=28)],
        },
        'date_features': ['dayofweek', 'month', 'quarter', 'week'],
        'num_threads': -1,
    },
    'fit': {
        'static_features': static_feature_cols,
    },
    'cv': {
        'h': h,
        'n_windows': 3,
        'step_size': h,
    },
}

def train_model(df, experiment_name):
    fcst = MLForecast(**params['init'])

    cv_results = fcst.cross_validation(
        df=df,
        id_col='unique_id',
        time_col='ds',
        target_col='y',
        **params['cv'],
        **params['fit'],
    )

    eval_result = evaluate(
        cv_results,
        metrics=[rmse, smape],
        agg_fn='mean',
    )

    logged_metrics = {}
    for _, row in eval_result.iterrows():
        metric = row['metric']
        for model in fcst.models_.keys() if hasattr(fcst, 'models_') and fcst.models_ else params['init']['models'].keys():
            logged_metrics[f'{metric}_{model}'] = float(row[model])

    logged_params = copy.deepcopy(params)
    logged_params['init']['models'] = {
        k: (v.__class__.__name__, v.get_params())
        for k, v in params['init']['models'].items()
    }
    logged_params['data_end_date'] = df['ds'].max().isoformat()

    with exp.start_run(experiment_name):
        exp.log_params(logged_params)
        exp.log_metrics(logged_metrics)
        fcst.fit(
            df,
            id_col='unique_id',
            time_col='ds',
            target_col='y',
            **params['fit'],
        )

    print('Cross-Validation Results:')
    display(eval_result)
    print(f'Features used: {fcst.ts.features_order_}')

    return fcst, logged_metrics, cv_results

# 7 - Train and Evaluate Model

Train the forecasting model using cross-validation and evaluate performance on the holdout set using RMSE and SMAPE metrics.

In [ ]:
fcst, metrics, cv_results = train_model(train, 'lgbm_mlforecast_run_1')

X_df = valid[['unique_id', 'ds'] + dynamic_feature_cols + operational_feature_cols]
preds = fcst.predict(h=h, X_df=X_df)
holdout_eval = evaluate(
    valid.merge(preds, on=['unique_id', 'ds']),
    metrics=[rmse, smape],
    agg_fn='mean',
)
print(f"CV RMSE: {metrics['rmse_LGBMRegressor']:.4f}")
print(f"Holdout RMSE: {holdout_eval.loc[holdout_eval['metric'] == 'rmse', 'LGBMRegressor'].item():.4f}")
print(f"Holdout SMAPE: {holdout_eval.loc[holdout_eval['metric'] == 'smape', 'LGBMRegressor'].item():.4f}")

# 8 - Feature Importance

Analyze which features contribute most to the model's predictions and visualize the forecast results against historical data for selected warehouses.

In [ ]:
lgbm_model = fcst.models_['LGBMRegressor']
feature_names = fcst.ts.features_order_
importances = lgbm_model.feature_importances_

importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values(by='importance', ascending=True)

demo_functions.plot_feature_importance(
    importance_df['feature'].tolist(),
    importance_df['importance'].tolist()
)

In [ ]:
demo_functions.plot_forecast(
    historical_df=train.rename(columns={'unique_id': 'WAREHOUSE_ID', 'ds': 'DATE', 'y': 'UTILIZATION_PCT'}),
    forecast_df=preds.rename(columns={'unique_id': 'WAREHOUSE_ID', 'ds': 'DATE', 'LGBMRegressor': 'FORECAST'}),
    warehouse_ids=['WH_001', 'WH_002', 'WH_003']
)

# 9 - Register Model in Model Registry

Wrap the mlforecast pipeline in a custom model class for deployment, then register it in the Snowflake Model Registry with metrics, signatures, and deployment targets.

In [ ]:
STATIC_FEATURE_COLS = ['IS_EMEA', 'IS_APAC', 'IS_AMERICAS', 'IS_COLD_STORAGE', 'IS_HAZMAT', 'IS_AMBIENT']
DYNAMIC_FEATURE_COLS = ['TOTAL_CAPACITY_SQM', 'NUM_LOADING_DOCKS', 'STAFFING_LEVEL']
OPERATIONAL_FEATURE_COLS = ['INBOUND_SHIPMENTS', 'OUTBOUND_SHIPMENTS', 'TEMPERATURE_C', 'NUM_ACTIVE_CLIENTS']
EXOG_FEATURE_COLS = DYNAMIC_FEATURE_COLS + OPERATIONAL_FEATURE_COLS

class WarehouseCapacityModel(CustomModel):
    def __init__(self, context: ModelContext) -> None:
        super().__init__(context)
        from mlforecast import MLForecast
        self.fcst = MLForecast.load(self.context['mlforecast_model'])

    @custom_model.partitioned_api
    def predict(self, input_df: pd.DataFrame) -> pd.DataFrame:
        import pandas as pd
        import warnings
        input_df = input_df.copy()
        input_df = input_df.rename(columns={
            'WAREHOUSE_ID': 'unique_id',
            'DATE': 'ds',
            'UTILIZATION_PCT': 'y',
        })
        input_df['ds'] = pd.to_datetime(input_df['ds'])

        history = input_df[input_df['y'].notna()].copy()
        future = input_df[input_df['y'].isna()].copy()

        dynamic_cols = [c for c in EXOG_FEATURE_COLS if c in input_df.columns]
        X_df = future[['unique_id', 'ds'] + dynamic_cols].copy()

        h = future.groupby('unique_id').size().max()

        input_ids = history['unique_id'].unique().tolist()

        with warnings.catch_warnings():
            warnings.filterwarnings('ignore', message='Found null values in')
            preds = self.fcst.predict(h=h, new_df=history, X_df=X_df, ids=input_ids)

        preds = preds.rename(columns={
            'unique_id': 'WAREHOUSE_ID',
            'ds': 'FORECAST_DATE',
            'LGBMRegressor': 'PREDICTED_UTILIZATION_PCT',
        })
        preds['FORECAST_DATE'] = preds['FORECAST_DATE'].dt.strftime('%Y-%m-%d')
        return preds[['FORECAST_DATE', 'PREDICTED_UTILIZATION_PCT']]

In [ ]:
import tempfile
import os
from snowflake.snowpark import Window

sf_feature_cols = ['WAREHOUSE_ID', 'DATE', 'UTILIZATION_PCT'] + all_feature_cols

def build_history_future_input(history_sf, h=demo_functions.FORECAST_HORIZON_DAYS):
    feature_cols = [F.col(c) for c in static_feature_cols + dynamic_feature_cols + operational_feature_cols]

    last_features_per_wh = (
        history_sf
        .with_column('_RN', F.row_number().over(
            Window.partition_by('WAREHOUSE_ID').order_by(F.col('DATE').desc())
        ))
        .filter(F.col('_RN') == 1)
        .select('WAREHOUSE_ID', *feature_cols)
    )

    max_date_per_wh = history_sf.group_by('WAREHOUSE_ID').agg(F.max('DATE').alias('MAX_DATE'))

    row_generator = session.generator(F.seq4().alias('SEQ'), rowcount=h)
    offsets = row_generator.select((F.col('SEQ') + 1).alias('DAY_OFFSET'))

    future_stubs = (
        max_date_per_wh
        .cross_join(offsets)
        .with_column('DATE', F.dateadd('day', F.col('DAY_OFFSET'), F.col('MAX_DATE')))
        .with_column('UTILIZATION_PCT', F.lit(None).cast('FLOAT'))
        .drop('MAX_DATE', 'DAY_OFFSET')
        .join(last_features_per_wh, on='WAREHOUSE_ID', how='left')
    )

    all_cols = ['WAREHOUSE_ID', 'DATE', 'UTILIZATION_PCT'] + static_feature_cols + dynamic_feature_cols + operational_feature_cols

    combined = (
        history_sf.select(all_cols)
        .union_all(future_stubs.select(all_cols))
        .with_column('DATE', F.col('DATE').cast('STRING'))
    )
    return combined

model_path = os.path.join(tempfile.mkdtemp(), 'mlforecast_model')
fcst.save(model_path)

model_context = ModelContext(mlforecast_model=model_path)
wh_model = WarehouseCapacityModel(model_context)

sample_input_snowpark = build_history_future_input(
    training_dataset_sf.select(sf_feature_cols).filter(F.col('WAREHOUSE_ID').isin(['WH_001', 'WH_002', 'WH_003']))
)
sample_input = sample_input_snowpark.to_pandas()

test_output = wh_model.predict(sample_input)
print(f'Input: {len(sample_input)} rows (history + future)')
print(f'Output: {len(test_output)} rows (predictions only)')
display(test_output.head(10))

In [ ]:
sample_input_sig = sample_input[sample_input['UTILIZATION_PCT'].notna()].copy()
sample_input_sig['DATE'] = sample_input_sig['DATE'].astype(str)

predict_signature = model_signature.infer_signature(
    input_data=sample_input_sig.head(),
    output_data=test_output.head()
)

registered_model = my_model_registry.log_model(
    wh_model,
    model_name="WAREHOUSE_CAPACITY_MODEL",
    version_name='V1',
    metrics=metrics,
    comment="mlforecast LightGBM pipeline for 90-day warehouse utilization forecasting",
    pip_requirements=['mlforecast', 'lightgbm'],
    signatures={'predict': predict_signature},
    artifact_repository_map={"pip": "snowflake.snowpark.pypi_shared_repository"},
    options={
        "method_options": {"predict": {"function_type": "TABLE_FUNCTION"}},
    },
    target_platforms=['SNOWPARK_CONTAINER_SERVICES', 'WAREHOUSE'],
    task=type_hints.Task.TABULAR_REGRESSION,
    sample_input_data=training_dataset_sf.select(sf_feature_cols).with_column('DATE', F.col('DATE').cast('STRING')).limit(100),
)

print('Model registered successfully.')

# 10 - Set Model to Production

Assign the PRODUCTION alias to the registered model and inspect its upstream lineage to verify feature store connections.

In [ ]:
registered_model.set_alias('PRODUCTION')

production_model = my_model_registry.get_model('WAREHOUSE_CAPACITY_MODEL').version('PRODUCTION')

In [ ]:
featureviews = production_model.lineage(direction='upstream')[0].lineage(domain_filter=['feature_view'], direction='upstream')
for featureview in featureviews:
    print(f'Feature View Name: {featureview.name}')
    print('Feature Names:')
    for feature in featureview.feature_names:
        print(f'  {feature}')

# 11 - Test Model

Run inference using the production model to generate baseline predictions for all warehouses over the forecast horizon.

In [ ]:
baseline_input_snowpark = build_history_future_input(training_dataset_sf.select(sf_feature_cols))

baseline_result = production_model.run(
    baseline_input_snowpark,
    function_name='predict',
    partition_column='WAREHOUSE_ID'
).cache_result()
baseline_predictions = baseline_result.select('WAREHOUSE_ID', 'FORECAST_DATE', 'PREDICTED_UTILIZATION_PCT')
print('Baseline predictions:')
baseline_predictions.show(10)

# 12 - Model Monitoring

Set up model monitoring by creating baseline and source tables, then configure a ModelMonitor to track prediction accuracy and detect drift over time.

In [ ]:
feature_cols_only = static_feature_cols + dynamic_feature_cols + operational_feature_cols

baseline_sf = (
    training_dataset_sf
    .select(
        'WAREHOUSE_ID',
        *[F.col(c) for c in feature_cols_only],
        F.col('UTILIZATION_PCT'),
        F.col('UTILIZATION_PCT').alias('PREDICTED_UTILIZATION_PCT'),
        F.col('DATE').cast('STRING').alias('FORECAST_DATE'),
        F.col('DATE').cast('TIMESTAMP').alias('TIMESTAMP'),
    )
)

baseline_sf.write.save_as_table(
    f'{database}.{schema}_MODEL_REGISTRY.WAREHOUSE_CAPACITY_MODEL_BASELINE_V1',
    mode='overwrite'
)

actuals_and_ops_sf = (
    session.table(f'{database}.{schema}.WAREHOUSE_OPERATIONS')
    .select(
        F.col('WAREHOUSE_ID').alias('ACT_WH_ID'),
        F.col('DATE').cast('STRING').alias('ACT_DATE'),
        F.col('UTILIZATION_PCT'),
        *[F.col(c).alias(f'OPS_{c}') for c in operational_feature_cols],
    )
)

latest_static_dynamic_sf = (
    training_dataset_sf
    .with_column('_RN', F.row_number().over(
        Window.partition_by('WAREHOUSE_ID').order_by(F.col('DATE').desc())
    ))
    .filter(F.col('_RN') == 1)
    .select(F.col('WAREHOUSE_ID').alias('FEAT_WH_ID'),
            *[F.col(c) for c in static_feature_cols + dynamic_feature_cols])
)

source_sf = (
    baseline_predictions
    .join(actuals_and_ops_sf, (F.col('WAREHOUSE_ID') == F.col('ACT_WH_ID')) &
                              (F.col('FORECAST_DATE') == F.col('ACT_DATE')), how='left')
    .drop('ACT_WH_ID', 'ACT_DATE')
    .join(latest_static_dynamic_sf, F.col('WAREHOUSE_ID') == F.col('FEAT_WH_ID'), how='left')
    .drop('FEAT_WH_ID')
)

for c in operational_feature_cols:
    source_sf = source_sf.with_column(c, F.col(f'OPS_{c}')).drop(f'OPS_{c}')

source_sf = source_sf.with_column('TIMESTAMP', F.col('FORECAST_DATE').cast('TIMESTAMP'))

source_sf.write.save_as_table(
    f'{database}.{schema}_MODEL_REGISTRY.WAREHOUSE_CAPACITY_MODEL_SOURCE_V1',
    mode='overwrite'
)

# 13 - Simulate Future Data and Model Predictions

Generate synthetic future data to simulate real-world operations, retrieve fresh features from the Feature Store, and merge actuals into the monitoring source table.

In [ ]:
simulation_start = (date.today() - timedelta(days=demo_functions.FORECAST_HORIZON_DAYS - 1)).isoformat()
simulation_end = (date.today() + timedelta(days=demo_functions.DRIFT_GRACE_DAYS)).isoformat()

demo_functions.generate_warehouse_data(
    session, schema,
    start_date=simulation_start,
    end_date=simulation_end,
    mode='append'
)

print(f'Simulated drift data from {simulation_start} to {simulation_end}')

In [ ]:
future_start = (date.today() - timedelta(days=demo_functions.FORECAST_HORIZON_DAYS + 30)).isoformat()

future_spine = (
    session.table(f'{database}.{schema}.WAREHOUSE_OPERATIONS')
    .filter(F.col('DATE') > future_start)
    .select('WAREHOUSE_ID', 'DATE', 'UTILIZATION_PCT')
    .distinct()
)

future_with_features = my_feature_store.retrieve_feature_values(
    spine_df=future_spine,
    features=[warehouse_static_fv, warehouse_dynamic_fv, warehouse_operational_fv],
    spine_timestamp_col='DATE',
)

future_input_snowpark = future_with_features.select(sf_feature_cols)

In [ ]:
source_table = f'{database}.{schema}_MODEL_REGISTRY.WAREHOUSE_CAPACITY_MODEL_SOURCE_V1'
actuals_table = f'{database}.{schema}.WAREHOUSE_OPERATIONS'

ops_update_cols = ', '.join([f's.{c} = a.{c}' for c in operational_feature_cols])

session.sql(f"""
    MERGE INTO {source_table} AS s
    USING (
        SELECT WAREHOUSE_ID, DATE::STRING AS FORECAST_DATE, UTILIZATION_PCT,
               {', '.join(operational_feature_cols)}
        FROM {actuals_table}
    ) AS a
    ON s.WAREHOUSE_ID = a.WAREHOUSE_ID AND s.FORECAST_DATE = a.FORECAST_DATE
    WHEN MATCHED THEN UPDATE SET s.UTILIZATION_PCT = a.UTILIZATION_PCT, {ops_update_cols}
""").collect()

print('Actuals and operational features merged into source table.')
session.table(source_table).select(
    F.count('*').alias('TOTAL_ROWS'),
    F.count('UTILIZATION_PCT').alias('ROWS_WITH_ACTUALS'),
).show()

In [ ]:
USE SCHEMA WAREHOUSE_OPTIMIZATION_MODEL_REGISTRY;

In [ ]:
source_config = ModelMonitorSourceConfig(
    baseline=f'{database}.{schema}_MODEL_REGISTRY.WAREHOUSE_CAPACITY_MODEL_BASELINE_V1',
    source=f'{database}.{schema}_MODEL_REGISTRY.WAREHOUSE_CAPACITY_MODEL_SOURCE_V1',
    timestamp_column='TIMESTAMP',
    id_columns=['WAREHOUSE_ID'],
    prediction_score_columns=['PREDICTED_UTILIZATION_PCT'],
    actual_score_columns=['UTILIZATION_PCT']
)

monitor_config = ModelMonitorConfig(
    model_version=production_model,
    model_function_name='predict',
    background_compute_warehouse_name=warehouse,
    refresh_interval='1 minute',
    aggregation_window='1 day'
)

model_monitor = my_model_registry.add_monitor(
    name=f'{database}.{schema}_MODEL_REGISTRY.WAREHOUSE_CAPACITY_MODEL_MM_V1',
    source_config=source_config,
    model_monitor_config=monitor_config
)

In [ ]:
source_table = f'{database}.{schema}_MODEL_REGISTRY.WAREHOUSE_CAPACITY_MODEL_SOURCE_V1'
future_pred_pd = session.table(source_table).select('WAREHOUSE_ID', 'FORECAST_DATE', 'PREDICTED_UTILIZATION_PCT').to_pandas()

demo_functions.plot_forecast(
    historical_df=ops_df.rename(columns={'unique_id': 'WAREHOUSE_ID', 'ds': 'DATE', 'y': 'UTILIZATION_PCT'}),
    forecast_df=future_pred_pd[['WAREHOUSE_ID', 'FORECAST_DATE', 'PREDICTED_UTILIZATION_PCT']].rename(columns={'FORECAST_DATE': 'DATE', 'PREDICTED_UTILIZATION_PCT': 'FORECAST'}),
    warehouse_ids=['WH_001', 'WH_002', 'WH_003']
)

# 14 - Retrain Model on Fresher Data

Create an updated training dataset that includes recent data, retrain the model, and register the new version. Promote V2 to production after verifying improved performance.

In [ ]:
training_cutoff_v2 = (date.today() - timedelta(days=demo_functions.FORECAST_HORIZON_DAYS - demo_functions.DRIFT_GRACE_DAYS)).isoformat()

spine_df_v2 = (
    session.table(f'{database}.{schema}.WAREHOUSE_OPERATIONS')
    .filter(F.col('DATE') <= training_cutoff_v2)
    .select('WAREHOUSE_ID', 'DATE', 'UTILIZATION_PCT')
    .distinct()
)

print(f'V2 training cutoff: {training_cutoff_v2}  (30 days after V1 cutoff: {training_cutoff})')

training_dataset_v2 = my_feature_store.generate_dataset(
    name=f'{database}.{schema}_MODEL_REGISTRY.WAREHOUSE_CAPACITY_DATASET_V2',
    spine_df=spine_df_v2,
    features=[warehouse_static_fv, warehouse_dynamic_fv, warehouse_operational_fv],
    spine_timestamp_col='DATE',
    spine_label_cols=['UTILIZATION_PCT'],
    desc='Immutable training dataset V2 — includes post-drift data for retraining.'
)

training_dataset_v2_sf = training_dataset_v2.read.to_snowpark_dataframe()
print(f'V2 Dataset created: {training_dataset_v2_sf.count()} rows')

In [ ]:
all_ops_df = training_dataset_v2_sf.order_by('WAREHOUSE_ID', 'DATE').to_pandas()

all_ops_df = all_ops_df.rename(columns={
    'WAREHOUSE_ID': 'unique_id',
    'DATE': 'ds',
    'UTILIZATION_PCT': 'y'
})
all_ops_df['ds'] = pd.to_datetime(all_ops_df['ds'])

if 'EFFECTIVE_DATE' in all_ops_df.columns:
    all_ops_df = all_ops_df.drop(columns=['EFFECTIVE_DATE'])
if 'DATE_1' in all_ops_df.columns:
    all_ops_df = all_ops_df.drop(columns=['DATE_1'])

print(f'Retrain data shape: {all_ops_df.shape}')
print(f'Date range: {all_ops_df["ds"].min()} to {all_ops_df["ds"].max()}')

In [ ]:
fcst_v2, metrics_v2, cv_results_v2 = train_model(all_ops_df, 'lgbm_mlforecast_run_2_retrained')

print(f'\nV1 RMSE: {metrics["rmse_LGBMRegressor"]:.4f}  →  V2 RMSE: {metrics_v2["rmse_LGBMRegressor"]:.4f}')
print(f'Improvement: {((metrics["rmse_LGBMRegressor"] - metrics_v2["rmse_LGBMRegressor"]) / metrics["rmse_LGBMRegressor"] * 100):.1f}%')

In [ ]:
model_path_v2 = os.path.join(tempfile.mkdtemp(), 'mlforecast_model_v2')
fcst_v2.save(model_path_v2)

model_context_v2 = ModelContext(mlforecast_model=model_path_v2)
wh_model_v2 = WarehouseCapacityModel(model_context_v2)

sample_input_v2_snowpark = build_history_future_input(
    training_dataset_v2_sf.select(sf_feature_cols).filter(F.col('WAREHOUSE_ID').isin(['WH_001', 'WH_002', 'WH_003']))
)
sample_input_v2 = sample_input_v2_snowpark.to_pandas()
test_output_v2 = wh_model_v2.predict(sample_input_v2)

predict_signature_v2 = model_signature.infer_signature(
    input_data=sample_input_v2[sample_input_v2['UTILIZATION_PCT'].notna()].head(),
    output_data=test_output_v2.head()
)

registered_model_v2 = my_model_registry.log_model(
    wh_model_v2,
    model_name="WAREHOUSE_CAPACITY_MODEL",
    version_name='V2',
    metrics=metrics_v2,
    comment="Retrained mlforecast LightGBM on full dataset including post-drift data",
    pip_requirements=['mlforecast', 'lightgbm'],
    artifact_repository_map={"pip": "snowflake.snowpark.pypi_shared_repository"},
    signatures={'predict': predict_signature_v2},
    options={
        "method_options": {"predict": {"function_type": "TABLE_FUNCTION"}},
    },
    target_platforms=['SNOWPARK_CONTAINER_SERVICES', 'WAREHOUSE'],
    task=type_hints.Task.TABULAR_REGRESSION,
    sample_input_data=training_dataset_v2_sf.select(sf_feature_cols).with_column('DATE', F.col('DATE').cast('STRING')).limit(100),
)

registered_model.unset_alias('PRODUCTION')
registered_model_v2.set_alias('PRODUCTION')

production_model = my_model_registry.get_model('WAREHOUSE_CAPACITY_MODEL').version('PRODUCTION')
print(f'PRODUCTION alias now points to: {production_model.version_name}')

# 15 - Model Monitoring V2

Set up monitoring infrastructure for the retrained model with updated baseline and source tables to track ongoing performance.

In [ ]:
baseline_v2_sf = (
    training_dataset_v2_sf
    .select(
        'WAREHOUSE_ID',
        *[F.col(c) for c in feature_cols_only],
        F.col('UTILIZATION_PCT'),
        F.col('UTILIZATION_PCT').alias('PREDICTED_UTILIZATION_PCT'),
        F.col('DATE').cast('STRING').alias('FORECAST_DATE'),
        F.col('DATE').cast('TIMESTAMP').alias('TIMESTAMP'),
    )
)

baseline_v2_sf.write.save_as_table(
    f'{database}.{schema}_MODEL_REGISTRY.WAREHOUSE_CAPACITY_MODEL_BASELINE_V2',
    mode='overwrite'
)
print(f'V2 baseline table: {baseline_v2_sf.count()} rows')

In [ ]:
v2_input_snowpark = build_history_future_input(training_dataset_v2_sf.select(sf_feature_cols))

v2_result = production_model.run(
    v2_input_snowpark,
    function_name='predict',
    partition_column='WAREHOUSE_ID'
).cache_result()

v2_predictions = v2_result.select('WAREHOUSE_ID', 'FORECAST_DATE', 'PREDICTED_UTILIZATION_PCT')

actuals_and_ops_v2_sf = (
    session.table(f'{database}.{schema}.WAREHOUSE_OPERATIONS')
    .select(
        F.col('WAREHOUSE_ID').alias('ACT_WH_ID'),
        F.col('DATE').cast('STRING').alias('ACT_DATE'),
        F.col('UTILIZATION_PCT'),
        *[F.col(c).alias(f'OPS_{c}') for c in operational_feature_cols],
    )
)

latest_static_dynamic_v2_sf = (
    training_dataset_v2_sf
    .with_column('_RN', F.row_number().over(
        Window.partition_by('WAREHOUSE_ID').order_by(F.col('DATE').desc())
    ))
    .filter(F.col('_RN') == 1)
    .select(F.col('WAREHOUSE_ID').alias('FEAT_WH_ID'),
            *[F.col(c) for c in static_feature_cols + dynamic_feature_cols])
)

source_v2_sf = (
    v2_predictions
    .join(actuals_and_ops_v2_sf, (F.col('WAREHOUSE_ID') == F.col('ACT_WH_ID')) &
                                 (F.col('FORECAST_DATE') == F.col('ACT_DATE')), how='left')
    .drop('ACT_WH_ID', 'ACT_DATE')
    .join(latest_static_dynamic_v2_sf, F.col('WAREHOUSE_ID') == F.col('FEAT_WH_ID'), how='left')
    .drop('FEAT_WH_ID')
)

for c in operational_feature_cols:
    source_v2_sf = source_v2_sf.with_column(c, F.col(f'OPS_{c}')).drop(f'OPS_{c}')

source_v2_sf = source_v2_sf.with_column('TIMESTAMP', F.col('FORECAST_DATE').cast('TIMESTAMP'))

source_v2_sf = source_v2_sf.filter(F.col('UTILIZATION_PCT').is_not_null())

source_v2_sf.write.save_as_table(
    f'{database}.{schema}_MODEL_REGISTRY.WAREHOUSE_CAPACITY_MODEL_SOURCE_V2',
    mode='overwrite'
)

print('V2 source table created (rows without actuals excluded to avoid null issues in monitor).')
session.table(f'{database}.{schema}_MODEL_REGISTRY.WAREHOUSE_CAPACITY_MODEL_SOURCE_V2').select(
    F.count('*').alias('TOTAL_ROWS'),
    F.count('UTILIZATION_PCT').alias('ROWS_WITH_ACTUALS'),
).show()

In [ ]:
source_config_v2 = ModelMonitorSourceConfig(
    baseline=f'{database}.{schema}_MODEL_REGISTRY.WAREHOUSE_CAPACITY_MODEL_BASELINE_V2',
    source=f'{database}.{schema}_MODEL_REGISTRY.WAREHOUSE_CAPACITY_MODEL_SOURCE_V2',
    timestamp_column='TIMESTAMP',
    id_columns=['WAREHOUSE_ID'],
    prediction_score_columns=['PREDICTED_UTILIZATION_PCT'],
    actual_score_columns=['UTILIZATION_PCT']
)

monitor_config_v2 = ModelMonitorConfig(
    model_version=production_model,
    model_function_name='predict',
    background_compute_warehouse_name=warehouse,
    refresh_interval='1 minute',
    aggregation_window='1 day'
)

model_monitor_v2 = my_model_registry.add_monitor(
    name=f'{database}.{schema}_MODEL_REGISTRY.WAREHOUSE_CAPACITY_MODEL_MM_V2',
    source_config=source_config_v2,
    model_monitor_config=monitor_config_v2
)

print(f'V2 monitor created: {model_monitor_v2}')

# 16 - Deploy Model as Inference Service in SPCS

Deploy the production model as a containerized inference service in Snowpark Container Services for scalable, real-time predictions.

In [ ]:
registered_model_v2.create_service(
    service_name="warehouse_capacity_prediction_service",
    service_compute_pool="SYSTEM_COMPUTE_POOL_CPU",
    ingress_enabled=True,
    gpu_requests=None
)

# 17 - Model Inference with Container Runtime

Run inference for a specific warehouse using the deployed model service to generate utilization forecasts.

In [ ]:
target_warehouse_id = 'WH_003'
rt_input = v2_input_snowpark.filter(F.col('WAREHOUSE_ID') == target_warehouse_id)

rt_result = production_model.run(
    rt_input, function_name='predict', partition_column='WAREHOUSE_ID'
)

print(f'{demo_functions.FORECAST_HORIZON_DAYS}-day Forecast for {target_warehouse_id}:')
rt_result.select(['WAREHOUSE_ID','FORECAST_DATE','PREDICTED_UTILIZATION_PCT']).order_by(F.col('FORECAST_DATE').desc()).limit(10)